# P7 — E: Paraphrase Attack Testing and Result Summary

**My Role**: Organize historical test results from A–D; prepare the paraphrased test set based on the new generator data provided by D; invoke the four existing detectors; save per-item scores, calculate post-attack metrics, and summarize the results. E did not train any new detectors.

**Actual Dataset**: 449 original passages, consisting of 150 human-written passages and 299 AI-generated passages. Post-attack remains 449 passages: the 150 human originals remain unchanged, while the 299 AI texts were each paraphrased, retaining their ground-truth label of 1 (AI).

**Workflow**: Original data → Fixed paraphrased test set → A/B/C/D score each passage → Compute metrics against ground-truth labels → Display side-by-side with members' historical results → Export four deliverable files.

By default, running this notebook only reads already completed real detection records and recalculates metrics; it does not download models, read API keys, or call paid APIs. The re-detection section at the end is disabled by default.

## 1. File Locations and Execution

After extracting the companion package, preserve the relative positions of `notebooks/` and `data/E/`. Alternatively, you can merge these two folders into the project root directory.
This notebook can be launched from either the project root directory or the `notebooks/` directory. Calculated results are written to `analysis/E_notebook_export/results/`, without overwriting the existing official `results/` or the original detection progress database.

Recalculating metrics requires `pandas`, `numpy`, and `scikit-learn`; charting uses `altair`. Re-invoking the detectors additionally requires the original project's dependencies, model files, and local configurations.

In [1]:
from pathlib import Path
from collections import Counter
import csv, hashlib, importlib, importlib.metadata, json, math, os, re, sqlite3, sys, time, tomllib
import xml.etree.ElementTree as ET
from zipfile import ZipFile
import numpy as np
import pandas as pd

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / 'data/E/after_scores.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Please extract the companion package first and open this notebook from inside the package or the project notebooks directory.')
DATA_DIR = ROOT / 'data/E'
EXPORT_DIR = ROOT / 'analysis/E_notebook_export/results'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
DETECTORS = list('ABCD')
NS = {'s': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
D_RESPONSE_ATTEMPTS = 4
D_MAX_TOKENS = 4096

def show_table(frame, digits=4):
    print(frame.to_string(index=False, float_format=lambda v: f'{v:.{digits}f}'))

print('Companion data found. Default mode: verify existing scores only, without re-invoking detectors.')


Companion data found. Default mode: verify existing scores only, without re-invoking detectors.


## 2. Labels, Scores, and Three Metrics

- `label/gold=0`: Original human text; `label/gold=1`: AI-sourced text. The AI label is retained after paraphrasing.
- `prob_ai`: The AI score returned by the detector; higher values indicate a stronger likelihood of being AI-generated. It is used for ranking; in particular, D's score is a judgment score given by an LLM and cannot be treated directly as a calibrated true probability.
- **AUROC**: The ability to rank AI texts higher than human texts; calculation requires both classes of text and continuous scores. 1 indicates perfect separation, 0.5 corresponds to random ranking; it is not "accuracy". This experiment does not invert scores below 0.5 to artificially boost performance.
- **F1**: Unified with AI as the positive class, classifying `score >= 0.5` as AI, and computed using `average="binary"`. The formula is `2TP/(2TP+FP+FN)`. F1 scores in the historical summary retain the members' original values and are not recalculated under a unified definition here.
- **TPR@1%FPR**: The proportion of AI texts detected when the false positive rate on human texts does not exceed 1%. Taken as the highest TPR among empirical ROC points where `FPR <= 0.01`, without interpolation. Out of 150 human texts, at most 1 can be misclassified as AI to meet this condition; this threshold is not necessarily 0.5.

In [2]:
def digest_bytes(value):
    return hashlib.sha256(value).hexdigest()

def text_digest(value):
    return digest_bytes(value.encode("utf-8"))

def file_digest(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def read_xlsx(path):
    """Read the first worksheet; no extra Excel-reading package is required."""
    with ZipFile(path) as z:
        shared = []
        if "xl/sharedStrings.xml" in z.namelist():
            shared = ["".join(n.itertext()) for n in ET.fromstring(
                z.read("xl/sharedStrings.xml")).findall("s:si", NS)]
        book = ET.fromstring(z.read("xl/workbook.xml"))
        first = book.find("s:sheets/s:sheet", NS)
        rel_id = first.attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]
        rels = ET.fromstring(z.read("xl/_rels/workbook.xml.rels"))
        target = next(r.attrib["Target"] for r in rels if r.attrib["Id"] == rel_id)
        if target.startswith("/"):
            target = target.lstrip("/")
        else:
            target = "xl/" + target
        grid = []
        for row in ET.fromstring(z.read(target)).findall("s:sheetData/s:row", NS):
            cells = {}
            for c in row.findall("s:c", NS):
                column = re.match(r"[A-Z]+", c.attrib["r"])[0]
                index = 0
                for char in column:
                    index = index * 26 + ord(char) - 64
                if c.find("s:f", NS) is not None:
                    raise ValueError("Input contains a formula; use fixed text/labels only.")
                v = c.find("s:v", NS)
                if c.attrib.get("t") == "s":
                    value = shared[int(v.text)]
                elif c.attrib.get("t") == "inlineStr":
                    value = "".join(c.find("s:is", NS).itertext())
                else:
                    value = v.text if v is not None else ""
                cells[index - 1] = value
            if any(cells.values()):
                grid.append([cells.get(i, "") for i in range(max(cells) + 1)])
    if not grid:
        raise ValueError("Empty input workbook")
    headers = [str(x).strip() for x in grid[0]]
    if len(set(headers)) != len(headers):
        raise ValueError("Duplicate column names")
    result = []
    for row in grid[1:]:
        obj = {h: str(row[i]) if i < len(row) else "" for i, h in enumerate(headers)}
        result.append(obj)
    return result

def validate_score(value):
    score = float(value)
    if not math.isfinite(score) or not 0 <= score <= 1:
        raise ValueError("Detector returned an invalid AI score")
    return score

def calculate_metrics(gold, scores):
    from sklearn.metrics import f1_score, roc_auc_score, roc_curve
    if set(gold) != {0, 1}:
        raise ValueError("AUROC requires both true classes")
    fpr, tpr, _ = roc_curve(gold, scores)
    return {"auroc": float(roc_auc_score(gold, scores)),
            "f1_binary_at_0_5": float(f1_score(gold, [int(v >= .5) for v in scores], average="binary", pos_label=1, zero_division=0)),
            "tpr_at_1fpr": max(float(t) for f, t in zip(fpr, tpr) if f <= .01 + 1e-12)}


## 3. Inspect Original Data and Paraphrased Test Set

The original AI texts were provided by D, comprising two sources: DeepSeek Chat and DeepSeek Reasoner. The paraphrased texts were generated with AI assistance and compiled into a fixed test set; these are not "human articles written from scratch".

Among the 299 AI passages, 100 underwent full-passage paraphrasing, while the remaining 199 primarily involved opening rephrasing, partially combined with sentence splitting, merging, lexical adjustments, and paragraph restructuring. Therefore, "all 299 passages were modified" does not mean every single sentence in every passage was completely rewritten.

The code below verifies each UID: alignment with original texts, whether human texts remain unchanged, whether AI texts were indeed modified, and whether ground-truth labels are accurate. Human/AI labels in the original file are mapped to 0/1 in memory without modifying the source file.

In [3]:
original = pd.read_csv(DATA_DIR / 'P7_new_generators.csv', encoding='utf-8-sig', dtype=str, keep_default_na=False)
original['label'] = original.label.map({'Human': '0', 'AI': '1', '0': '0', '1': '1'})
assert original.label.notna().all()
after = pd.DataFrame(read_xlsx(DATA_DIR / 'attack_after_449.xlsx'))
metadata = json.loads((DATA_DIR / 'run_metadata.json').read_text(encoding='utf-8-sig'))
assert len(original) == len(after) == 449
assert original.uid.is_unique and after.uid.is_unique
original_index = original.set_index('uid')
after['original_uid'] = after['uid'].str.removeprefix('paraphrased_')
assert set(after.original_uid) == set(original.uid)
assert set(after.label) == {'0', '1'}
for row in after.to_dict('records'):
    source = original_index.loc[row['original_uid']]
    assert row['label'] == source['label']
    assert row['prompt_id'] == source['prompt_id']
    assert row['text'].strip()
    if row['label'] == '0':
        assert row['condition'] == 'human' and row['text'] == source['text']
    else:
        assert row['condition'] == 'after' and row['text'] != source['text']
        assert row['original_source_model'] == source['source_model']
assert file_digest(DATA_DIR / 'attack_after_449.xlsx') == metadata['after_sha256']
human_n = int((after.label == '0').sum())
ai_n = int((after.label == '1').sum())
assert (human_n, ai_n) == (150, 299)
print(f'Verification passed: total {len(after)} passages; {human_n} original human passages; {ai_n} paraphrased AI passages.')
print('\nOriginal AI sources:')
show_table(original.loc[original.label == '1', 'source_model'].value_counts().rename_axis('source_model').reset_index(name='n'))
print('\nRewriting methods (original records retained for combined methods):')
methods = after.loc[after.label == '1', 'rewrite_method'].value_counts().rename_axis('rewrite_method').reset_index(name='n')
show_table(methods)
full_n = int((after.loc[after.label == '1', 'rewrite_method'] == 'full_passage_paraphrase').sum())
assert full_n == 100
print(f'Full-passage paraphrase: {full_n} passages; primarily opening rephrase and combined methods: {ai_n - full_n} passages.')


Verification passed: total 449 passages; 150 original human passages; 299 paraphrased AI passages.

Original AI sources:
     source_model   n
    deepseek-chat 150
deepseek-reasoner 149

Rewriting methods (original records retained for combined methods):
                                                                      rewrite_method   n
                                                                    opening_rephrase 104
                                                             full_passage_paraphrase 100
                                               opening_rephrase;paragraph_restructure  27
                                                     opening_rephrase;sentence_merge  19
                                                     opening_rephrase;sentence_split  11
                                opening_rephrase;sentence_merge;paragraph_restructure  10
                                     opening_rephrase;lexical_or_connective_variation  10
               opening_rephra

## 4. Load Actual Per-Item Detection Records

`after_scores.csv` contains saved records from actual runs, not dummy scores. Each detector must fully cover the same 449 test passages, totaling 1,796 successful records across all four detectors.
Detectors only receive `text` and are not given ground-truth labels, `source_model`, or hints like "paraphrased". Below, scores, labels, and text SHA-256 hashes are verified item by item to prevent mixing up results from other files.

In [4]:
scores = pd.read_csv(DATA_DIR / 'after_scores.csv', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert len(scores) == 4 * len(after) == 1796
assert not scores.duplicated(['detector', 'uid']).any()
assert set(scores.detector) == set(DETECTORS)
assert set(scores.status) == {'ok'}
scores['prob_ai'] = pd.to_numeric(scores['prob_ai'], errors='raise')
scores['gold'] = pd.to_numeric(scores['gold'], errors='raise').astype(int)
assert np.isfinite(scores.prob_ai).all() and scores.prob_ai.between(0, 1).all()
target = after.set_index('uid')
for row in scores.to_dict('records'):
    record = target.loc[row['uid']]
    assert row['gold'] == int(record['label'])
    assert row['condition'] == record['condition']
    assert row['text_sha256'] == text_digest(record['text'])
    assert int(row['pred_at_0_5']) == int(row['prob_ai'] >= 0.5)
for detector in DETECTORS:
    assert set(scores.loc[scores.detector == detector, 'uid']) == set(after.uid)
coverage = scores.groupby('detector').agg(n=('uid', 'size'), ai_n=('gold', 'sum')).reset_index()
coverage['human_n'] = coverage.n - coverage.ai_n
print('Per-item detection records verification passed:')
show_table(coverage[['detector', 'n', 'human_n', 'ai_n']])


Per-item detection records verification passed:
detector   n  human_n  ai_n
       A 449      150   299
       B 449      150   299
       C 449      150   299
       D 449      150   299


## 5. Recalculate Post-Attack Metrics from Per-Item Scores

This step only computes statistics and does not re-run models or query DeepSeek. Finally, the recalculated results are cross-checked item by item against `after_metrics_recorded.csv` saved by the original detection script.

In [5]:
metric_rows = []
for detector in DETECTORS:
    part = scores[scores.detector == detector]
    values = calculate_metrics(part.gold.tolist(), part.prob_ai.tolist())
    metric_rows.append({'detector': detector, 'n': len(part),
                        'human_n': int((part.gold == 0).sum()),
                        'ai_n': int((part.gold == 1).sum()), **values})
metrics = pd.DataFrame(metric_rows)
recorded = pd.read_csv(DATA_DIR / 'after_metrics_recorded.csv', encoding='utf-8-sig').set_index('detector')
for row in metric_rows:
    for key, value in row.items():
        if key != 'detector':
            assert math.isclose(float(value), float(recorded.loc[row['detector'], key]), rel_tol=0, abs_tol=1e-12)
metrics.to_csv(EXPORT_DIR.parent / 'after_metrics_recomputed.csv', index=False, encoding='utf-8-sig')
print('Recalculated results match original script output (absolute difference <= 1e-12):')
show_table(metrics)


Recalculated results match original script output (absolute difference <= 1e-12):
detector   n  human_n  ai_n  auroc  f1_binary_at_0_5  tpr_at_1fpr
       A 449      150   299 0.7345            0.7877       0.0167
       B 449      150   299 0.2841            0.0189       0.0000
       C 449      150   299 0.5572            0.5000       0.1204
       D 449      150   299 0.9692            0.9233       0.5084


## 6. Summarize Historical Results and Define Comparison Boundaries

Main results for HC3, MAGE, RAID, and New generators retain the compiled scores uploaded by members, without retraining or recalculating historical F1. `auroc_reported.csv` contains the 16 compiled main results, preserving the numerical precision submitted by users.

Pre-attack uses members' historical results, while post-attack uses the newly detected 449 results from this run. Historical per-item files show that sample counts for New generators across A, B, C, and D were not completely identical; furthermore, D increased its reply token limit from 400 in the original interface to 4096 in this run and added bounded retries for failed requests. Therefore, this is a side-by-side comparison of historical and post-attack results, and not all differences can be attributed solely to paraphrasing.

In [6]:
auroc = pd.read_csv(DATA_DIR / 'auroc_reported.csv', encoding='utf-8-sig', dtype=str, keep_default_na=False)
expected_header = ['dataset', 'detector', 'auroc', 'f1', 'tpr_at_1fpr']
assert list(auroc.columns) == expected_header
assert len(auroc) == 16 and not auroc.duplicated(['dataset', 'detector']).any()
assert set(auroc.dataset) == {'HC3', 'MAGE', 'RAID', 'New generators'}
historical_coverage = []
for detector in DETECTORS:
    member = pd.read_csv(DATA_DIR / 'member_results' / f'P7_{detector}_preds.csv', dtype=str, keep_default_na=False, encoding='utf-8-sig')
    is_new = member.dataset.str.lower().str.replace('_', ' ', regex=False).isin(['new generators', 'newgen'])
    part = member[is_new]
    historical_coverage.append({'detector': detector, 'before_n': len(part),
        'human_n_before': int((part.gold == '0').sum()), 'ai_n_before': int((part.gold == '1').sum()), 'after_n': 449})
print('Historical New generators per-item record coverage:')
show_table(pd.DataFrame(historical_coverage))
before_values = auroc[auroc.dataset == 'New generators'][['detector', 'auroc']].rename(columns={'auroc': 'auroc_before'})
comparison = before_values.merge(metrics[['detector', 'auroc']], on='detector', validate='one_to_one').rename(columns={'auroc': 'auroc_after'})
comparison['auroc_before'] = comparison.auroc_before.astype(float)
comparison['delta_after_minus_before'] = comparison.auroc_after - comparison.auroc_before
print('\nSide-by-side AUROC comparison (negative values indicate lower post-attack scores in current records; does not imply other factors are excluded):')
show_table(comparison)


Historical New generators per-item record coverage:
detector  before_n  human_n_before  ai_n_before  after_n
       A       420             121          299      449
       B       449             150          299      449
       C       449             150          299      449
       D       445             149          296      449

Side-by-side AUROC comparison (negative values indicate lower post-attack scores in current records; does not imply other factors are excluded):
detector  auroc_before  auroc_after  delta_after_minus_before
       A        0.7600       0.7345                   -0.0255
       B        0.4700       0.2841                   -0.1859
       C        0.6100       0.5572                   -0.0528
       D        0.9500       0.9692                    0.0192


## 7. Plot Pre- vs. Post-Attack Comparison Chart

The y-axis is fixed to 0–1 to fully display B's low score. This chart illustrates the historical vs. current score comparison from the previous section; colors distinguish before/after stages rather than human/AI text types.

In [7]:
import altair as alt
chart_data = comparison[['detector', 'auroc_before', 'auroc_after']].melt(
    id_vars='detector', var_name='stage', value_name='auroc')
chart_data['stage'] = chart_data.stage.map({'auroc_before': 'Historical before', 'auroc_after': 'After rewriting'})
chart = alt.Chart(chart_data).mark_bar().encode(
    x=alt.X('detector:N', title='Detector', sort=DETECTORS),
    xOffset=alt.XOffset('stage:N', sort=['Historical before', 'After rewriting']),
    y=alt.Y('auroc:Q', title='AUROC', scale=alt.Scale(domain=[0, 1])),
    color=alt.Color('stage:N', title='Evaluation', scale=alt.Scale(
        domain=['Historical before', 'After rewriting'], range=['#777777', '#e88930'])),
    tooltip=['detector:N', 'stage:N', alt.Tooltip('auroc:Q', format='.4f')]
).properties(width=560, height=300, title='Historical baseline vs. after rewriting')
CHART_SPEC = chart.to_dict()
chart.save(EXPORT_DIR.parent / 'P7_E_attack_auroc.html')
(EXPORT_DIR.parent / 'P7_E_attack_auroc.vl.json').write_text(json.dumps(CHART_SPEC, ensure_ascii=False), encoding='utf-8')
print('Comparison chart generated; HTML chart can be viewed in a browser (internet connection required to load interactive components).')
try:
    from IPython.display import display
except ImportError:
    pass
else:
    display(chart)


Comparison chart generated; HTML chart can be viewed in a browser (internet connection required to load interactive components).


AUROC chart: historical baseline vs. after rewriting; y-axis 0–1.

## 8. Verify Demo Pairs and Export Four Deliverable Files

The four pairs of original and paraphrased texts in `attack_pairs.csv` are taken from actual test passages rather than newly fabricated demo articles. They are verified character by character here before exporting to a separate directory.

Main results in `auroc.csv` are preserved as-is. In the attack comparison, `auroc_before` retains historical summary values, while post-attack values are recalculated from the 1,796 records and saved at full precision (can be rounded to two decimal places for display). Decimal places in the two CSV files may differ from manually rounded submission versions, without altering the raw detector scores.

In [8]:
pairs = pd.read_csv(DATA_DIR / 'attack_pairs_reported.csv', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert {'id', 'uid', 'prompt_id', 'original', 'attacked', 'attack_type'} <= set(pairs.columns)
assert 3 <= len(pairs) <= 5 and pairs.id.is_unique and pairs.uid.is_unique
after_index = after.set_index('original_uid')
for pair in pairs.to_dict('records'):
    assert pair['original'] == original_index.loc[pair['uid'], 'text']
    assert pair['attacked'] == after_index.loc[pair['uid'], 'text']
    assert pair['prompt_id'] == original_index.loc[pair['uid'], 'prompt_id']
    assert original_index.loc[pair['uid'], 'label'] == '1'
attack = before_values.merge(metrics, on='detector', validate='one_to_one').rename(columns={
    'auroc': 'auroc_after', 'f1_binary_at_0_5': 'f1_binary_at_0_5_after',
    'tpr_at_1fpr': 'tpr_at_1fpr_after', 'n': 'total',
    'human_n': 'human_Original', 'ai_n': 'ai_Rewritten'})
attack = attack[['detector', 'auroc_before', 'auroc_after', 'f1_binary_at_0_5_after',
                 'tpr_at_1fpr_after', 'total', 'human_Original', 'ai_Rewritten']]
auroc.to_csv(EXPORT_DIR / 'auroc.csv', index=False, encoding='utf-8-sig')
attack.to_csv(EXPORT_DIR / 'attack.csv', index=False, encoding='utf-8-sig')
pairs.to_csv(EXPORT_DIR / 'attack_pairs.csv', index=False, encoding='utf-8-sig')
indexed = comparison.set_index('detector')
d_result = metrics.set_index('detector').loc['D']
findings = [
    'Based on current results, ' + ', '.join(f'the AUROC of {d} changed from {indexed.loc[d, "auroc_before"]:.2f} to {indexed.loc[d, "auroc_after"]:.2f}' for d in 'ABC') + ', with B showing the largest drop.',
    'After the attack, B had a TPR@1%FPR of 0, indicating that in this test, no AI texts were detected when the false positive rate on human texts was constrained within 1%.',
    f'The post-attack set contains 150 unrewritten human texts and 299 rewritten AI texts; D performed best with an AUROC of {d_result.auroc:.2f}, F1 (binary, threshold 0.5) of {d_result.f1_binary_at_0_5:.2f}, and TPR@1%FPR of {d_result.tpr_at_1fpr:.2f}.',
    'Pre-attack metrics use members\' original results while post-attack metrics use newly detected results; sample sizes and runtime settings were not completely unified, so not all differences can be attributed solely to paraphrasing.'
]
(EXPORT_DIR / 'findings.txt').write_text('\n'.join(findings) + '\n', encoding='utf-8-sig')
for name in ['auroc.csv', 'attack.csv', 'attack_pairs.csv']:
    check = pd.read_csv(EXPORT_DIR / name, encoding='utf-8-sig')
    print(f'{name}: {len(check)} records, CSV is readable.')
print('findings.txt: 4 findings.')
print('Output location: analysis/E_notebook_export/results; original results have not been overwritten.')


auroc.csv: 16 records, CSV is readable.
attack.csv: 4 records, CSV is readable.
attack_pairs.csv: 4 records, CSV is readable.
findings.txt: 4 findings.
Output location: analysis/E_notebook_export/results; original results have not been overwritten.


## 9. Optional: Full Code to Re-run Real Detectors

**Detection has already been completed for this run; no new detection needs to be executed.** The code below provides a reproducible entry point for the experimental method, with `RUN_NEW_DETECTION=False` by default so no requests will be sent.

A calls `predict()`; B calls `predict()`; C calls `score_texts()` in batches of 8 passages to obtain passage-level scores; D uses the member's model, prompt, and normalization function with `max_tokens=4096`, up to 4 outer attempts, preserving native SDK retries, and never filling failed API calls with 0. D's reasoning/thinking settings follow server defaults, so future changes in models or service defaults may lead to different results.

Re-running requires local API key configuration, incurs API costs for D, and may download models on the first run. Do not write keys into the notebook or upload them. New detections use a separate directory and will not write to the previously completed `progress.sqlite3`. The original 1,796 scores are always used in the official verification section above and will not be silently overwritten by new runs.

In [9]:
def load_local_settings(project):
    """Never print, copy, or write credentials to output files."""
    merged = {}
    for path in (Path.home() / ".streamlit/secrets.toml", project / ".streamlit/secrets.toml"):
        if path.exists():
            try:
                settings = tomllib.loads(path.read_text(encoding="utf-8-sig"))
                merged.update({k: v for k, v in settings.items() if isinstance(v, str)})
            except tomllib.TOMLDecodeError:
                raise ValueError("The local secrets.toml is not valid TOML; its contents are not shown.") from None
    for key, value in merged.items():
        os.environ.setdefault(key, value)
    try:
        from dotenv import load_dotenv
        load_dotenv(project / ".env", override=False)
    except ImportError:
        pass
    mode = os.environ.get("P7_D_PROMPT", "few_shot")
    if mode == "few_shot" and not os.environ.get("P7_D_FEWSHOT"):
        candidates = [project / "data/fewshot_examples.json",
                      project / "D_others/D to ABC/fewshot_examples.json",
                      project / "D_others/D  to F/fewshot_examples.json"]
        found = next((p for p in candidates if p.is_file()), None)
        if found:
            os.environ["P7_D_FEWSHOT"] = str(found)
    return mode

def safe_error(exc):
    text = f"{type(exc).__name__}: {exc}"
    for key in ("DEEPSEEK_API_KEY", "P7_API_KEY", "OPENAI_API_KEY"):
        secret = os.environ.get(key)
        if secret:
            text = text.replace(secret, "[redacted]")
    return re.sub(r"\bsk-[A-Za-z0-9_-]+", "[redacted]", text)[:1200]

def record(conn, detector, task, status, score=None, native_label="", elapsed=0.0, error=""):
    conn.execute("INSERT OR REPLACE INTO scores VALUES (?,?,?,?,?,?,?,?,?)", (
        detector, task["uid"], task["text_sha256"], status, score, native_label,
        elapsed, error, time.strftime("%Y-%m-%dT%H:%M:%S")))
    conn.commit()

def request_D_json(module,text):
    """Use the member's exact prompt/model with a sufficient, fixed reply budget."""
    cleaned=(module.normalize_text(text) if module.NORMALIZE else text.strip())[:module.MAX_CHARS]
    response=module._get_client().chat.completions.create(
        model=module.MODEL,temperature=0,max_tokens=D_MAX_TOKENS,
        response_format={"type":"json_object"},
        messages=[{"role":"system","content":module.SYSTEM_PROMPT},
                  {"role":"user","content":module._build_user_prompt(cleaned,module.PROMPT_MODE)}])
    if not response.choices:
        raise RuntimeError("Empty response: no choices")
    choice=response.choices[0]
    finish=getattr(choice,"finish_reason",None)
    content=(choice.message.content or "").strip()
    usage=getattr(response,"usage",None)
    completion_tokens=getattr(usage,"completion_tokens",None)
    if not content or finish=="length":
        print(f"[D] reply diagnostics: finish={finish}, content_chars={len(content)}, completion_tokens={completion_tokens}, max_tokens={D_MAX_TOKENS}",flush=True)
    if finish=="length":
        raise RuntimeError(f"Truncated response at max_tokens={D_MAX_TOKENS}")
    if not content:
        raise RuntimeError(f"Empty response (finish={finish}, completion_tokens={completion_tokens})")
    result=json.loads(content)
    value=result.get("ai_score") if isinstance(result,dict) else None
    if isinstance(value,bool) or not isinstance(value,(int,float)) or not math.isfinite(value) or not 0<=value<=100:
        raise ValueError("D returned an invalid ai_score")
    return {"ai_score":value,"reason":str(result.get("reason","")),"finish_reason":finish,
            "completion_tokens":completion_tokens}

def judge_with_response_retries(module, text, attempts=D_RESPONSE_ATTEMPTS):
    """Bounded identical-request retries; never score partial JSON or reasoning text."""
    for attempt in range(1, attempts + 1):
        try:
            return request_D_json(module,text)
        except Exception as exc:
            status = getattr(exc,"status_code",None)
            if status in (400,401,402,403,404) or type(exc).__name__=="BalanceError":
                raise
            transient = (isinstance(exc,(json.JSONDecodeError,TimeoutError,ConnectionError))
                         or (isinstance(exc,RuntimeError) and any(s in str(exc).lower() for s in ("empty response","truncated response")))
                         or status==429 or (isinstance(status,int) and status>=500)
                         or type(exc).__name__ in {"APITimeoutError","APIConnectionError","RateLimitError","InternalServerError"})
            if not transient or attempt==attempts:
                raise
            delay=min(2**(attempt-1),8)
            print(f"[D] incomplete/temporary reply; retry {attempt+1}/{attempts} in {delay}s",flush=True)
            time.sleep(delay)

def run_detector(conn, project, detector, tasks, limit, batch_size):
    selected = tasks[:limit] if limit else tasks
    completed = {uid: sha for uid, sha in conn.execute(
        "SELECT uid,text_sha256 FROM scores WHERE detector=? AND status='ok'", (detector,))}
    pending = [r for r in selected if completed.get(r["uid"]) != r["text_sha256"]]
    print(f"\n[{detector}] selected={len(selected)}, cached={len(selected)-len(pending)}, remaining={len(pending)}", flush=True)
    if not pending:
        return
    # Direct member interfaces only. Never use the front end's mock fallback.
    module = importlib.import_module(f"P7_{detector}_predict")
    failures = 0
    started = time.monotonic()
    chunks = [pending[i:i + batch_size] for i in range(0, len(pending), batch_size)] if detector == "C" else [[r] for r in pending]
    for chunk in chunks:
        if detector == "C":
            tick = time.monotonic()
            try:
                result = module.score_texts([r["text"] for r in chunk], batch_size=batch_size)
                if len(result) != len(chunk):
                    raise ValueError("C returned the wrong number of scores")
                result = [validate_score(v) for v in result]
                per_text = (time.monotonic() - tick) / len(chunk)
                for task, score in zip(chunk, result):
                    record(conn, detector, task, "ok", score, "AI" if score >= .5 else "Human", per_text)
                failures = 0
            except Exception as exc:
                for task in chunk:
                    record(conn, detector, task, "error", error=safe_error(exc))
                failures += len(chunk)
                print(f"[{detector}] ERROR: {safe_error(exc)}", flush=True)
        else:
            task = chunk[0]
            tick = time.monotonic()
            try:
                if detector == "D":
                    r = judge_with_response_retries(module,task["text"])
                    score = validate_score(r["ai_score"] / 100)
                    native = "AI" if score >= module.THRESHOLD else "Human"
                else:
                    r = module.predict(task["text"])
                    if not isinstance(r, dict) or "prob_ai" not in r:
                        raise ValueError("Detector returned no score")
                    if (r.get("extra") or {}).get("note"):
                        raise ValueError("Detector marked passage unusable: " + str(r["extra"]["note"]))
                    score = validate_score(r["prob_ai"])
                    native = str(r.get("label", ""))
                record(conn, detector, task, "ok", score, native, time.monotonic() - tick)
                failures = 0
            except Exception as exc:
                record(conn, detector, task, "error", error=safe_error(exc), elapsed=time.monotonic()-tick)
                failures += 1
                print(f"[{detector}] {task['uid']} ERROR: {safe_error(exc)}", flush=True)
                if getattr(exc, "status_code", None) in (401, 402, 403) or type(exc).__name__ == "BalanceError":
                    raise RuntimeError("API authentication, access, or balance error. Progress was saved.") from None
        ok_ids = {r[0] for r in conn.execute("SELECT uid FROM scores WHERE detector=? AND status='ok'", (detector,))}
        total = sum(r["uid"] in ok_ids for r in tasks)
        print(f"[{detector}] saved {total}/{len(tasks)} | {time.monotonic()-started:.1f}s this session", flush=True)
        if failures >= 3:
            raise RuntimeError(f"{detector}: repeated failures. Stopped instead of filling failures with zero.")


In [10]:
RUN_NEW_DETECTION = False
# Modify only when re-detection is actually needed. The original project is typically located at AI-Detector-main on Desktop.
PROJECT_DIR = ROOT if (ROOT / 'P7_A_predict.py').is_file() else Path.home() / 'Desktop/AI-Detector-main'
NEW_RUN_DIR = ROOT / 'analysis/E_notebook_new_detection'

def open_new_run_cache(path, fingerprint):
    conn = sqlite3.connect(path)
    conn.execute('CREATE TABLE IF NOT EXISTS metadata (key TEXT PRIMARY KEY, value TEXT NOT NULL)')
    conn.execute("""CREATE TABLE IF NOT EXISTS scores (
        detector TEXT, uid TEXT, text_sha256 TEXT NOT NULL, status TEXT NOT NULL,
        prob_ai REAL, native_label TEXT, elapsed_seconds REAL, error TEXT,
        updated_at TEXT, PRIMARY KEY(detector,uid))""")
    serial = json.dumps(fingerprint, sort_keys=True, ensure_ascii=False)
    saved = conn.execute("SELECT value FROM metadata WHERE key='fingerprint'").fetchone()
    if saved and saved[0] != serial:
        conn.close()
        raise ValueError('Inputs, code, settings, or dependencies for the new detection have changed; please use a new NEW_RUN_DIR and do not mix scores.')
    conn.execute('INSERT OR IGNORE INTO metadata VALUES (?, ?)', ('fingerprint', serial))
    conn.commit()
    return conn

def run_optional_new_detection():
    project = PROJECT_DIR.resolve()
    for d in DETECTORS:
        if not (project / f'P7_{d}_predict.py').is_file():
            raise FileNotFoundError(f'Please confirm the locations of PROJECT_DIR and P7_{d}_predict.py.')
    mode = load_local_settings(project)
    if mode != 'few_shot':
        raise ValueError('This experiment uses few_shot; current setting differs, please verify first.')
    if not (os.environ.get('P7_API_KEY') or os.environ.get('DEEPSEEK_API_KEY')):
        raise ValueError('Please configure D\'s API key locally; do not write it into the notebook.')
    shot_path = Path(os.environ.get('P7_D_FEWSHOT', ''))
    if not shot_path.is_file():
        raise ValueError('Missing real few-shot examples file.')
    shots = json.loads(shot_path.read_text(encoding='utf-8-sig'))
    if not shots or any(str(s.get('qid', '')) in set(after.prompt_id) for s in shots if s.get('qid')):
        raise ValueError('few-shot examples are empty or overlap with test prompts.')
    if file_digest(shot_path) != metadata['settings']['fewshot_sha256']:
        raise ValueError('few-shot examples do not match the original experiment.')
    settings = {key: os.environ.get(key, default) for key, default in {
        'P7_D_MODEL': 'deepseek-flash', 'P7_D_PROMPT': 'few_shot',
        'P7_D_THRESHOLD': '0.50', 'P7_D_NORMALIZE': '1', 'P7_B_MODEL': 'gpt2',
        'P7_C_MODEL': 'Hanxi79/p7-detector', 'P7_B_CALIBRATOR_REPO': 'captainfan/p7-gpt2-gltr-detector'}.items()}
    settings['base_url_sha256'] = text_digest(os.environ.get('P7_BASE_URL', 'https://api.deepseek.com'))
    settings['fewshot_sha256'] = file_digest(shot_path)
    if os.environ.get('P7_B_CALIBRATOR'):
        settings['B_local_classifier_sha256'] = file_digest(Path(os.environ['P7_B_CALIBRATOR']))
    if settings != metadata['settings']:
        raise ValueError('Current detector settings do not match the original experiment; verify models and configurations first.')
    module_paths = [project / f'P7_{d}_predict.py' for d in DETECTORS] + [project / 'models/P7_A_model.pkl']
    current_files = {str(p.relative_to(project)): file_digest(p) for p in module_paths if p.is_file()}
    if current_files != metadata['detector_files']:
        raise ValueError('Detector or A model files have changed; verify versions first.')
    fingerprint = {'after_sha256': file_digest(DATA_DIR / 'attack_after_449.xlsx'),
        'detector_files': current_files, 'settings': settings,
        'versions': {p: importlib.metadata.version(p) for p in ['numpy', 'scikit-learn', 'xgboost', 'torch', 'transformers', 'openai']},
        'D_max_tokens': D_MAX_TOKENS, 'D_response_attempts': D_RESPONSE_ATTEMPTS,
        'code_sha256': hashlib.sha256(run_detector.__code__.co_code + request_D_json.__code__.co_code + judge_with_response_retries.__code__.co_code).hexdigest()}
    tasks = [dict(r, text_sha256=text_digest(r['text'])) for r in after.to_dict('records')]
    NEW_RUN_DIR.mkdir(parents=True, exist_ok=True)
    conn = open_new_run_cache(NEW_RUN_DIR / 'progress.sqlite3', fingerprint)
    old_cwd = Path.cwd()
    old_path = list(sys.path)
    try:
        sys.path.insert(0, str(project))
        os.chdir(project)
        if any(f'P7_{d}_predict' in sys.modules for d in DETECTORS):
            raise RuntimeError('Please restart the notebook kernel before re-detecting to prevent reusing cached old modules.')
        for detector in DETECTORS:
            run_detector(conn, project, detector, tasks, limit=0, batch_size=8)
    finally:
        os.chdir(old_cwd)
        sys.path[:] = old_path
        saved = pd.read_sql_query('SELECT * FROM scores ORDER BY detector, uid', conn)
        saved.to_csv(NEW_RUN_DIR / 'new_scores.csv', index=False, encoding='utf-8-sig')
        (NEW_RUN_DIR / 'metadata.json').write_text(json.dumps(fingerprint, ensure_ascii=False, indent=2), encoding='utf-8-sig')
        conn.close()
    complete = len(saved) == 4 * len(after) and set(saved.status) == {'ok'}
    if complete:
        new_rows = []
        for d in DETECTORS:
            subset = saved[saved.detector == d]
            labels = [int(target.loc[uid, 'label']) for uid in subset.uid]
            new_rows.append({'detector': d, **calculate_metrics(labels, subset.prob_ai.tolist())})
        pd.DataFrame(new_rows).to_csv(NEW_RUN_DIR / 'new_metrics.csv', index=False, encoding='utf-8-sig')
        print('New detection completed; saved separate results without replacing original experiment scores.')
    else:
        print('New detection not yet fully completed; existing progress retained, final new metrics not generated.')

if RUN_NEW_DETECTION:
    run_optional_new_detection()
else:
    print('Skipped re-detection: this run only verifies original real scores without making API requests.')


Skipped re-detection: this run only verifies original real scores without making API requests.


## 10. Experimental Notes, Limitations, and References

- A–D were not retrained in this experiment; they were only scored using existing interfaces. Task E should not be described as "training four models".
- The fixed paraphrased texts serve as experimental inputs and are not randomly regenerated during default execution, ensuring local verification uses identical texts. Paraphrasing intensity varies across the 299 passages, so it cannot be inferred that all attack types and intensities yield the same effect.
- Original texts come from D's fixed test set. Sample sizes for New generators in the historical summary were A=420, B=449, C=449, and D=445, whereas post-attack sample sizes are 449 for all. Only reported metrics are compared, without claiming strictly paired causal effects.
- Version mismatch warnings for `scikit-learn` occurred when loading original A/B models; D's online model aliases and server-side defaults may also change over time. Historical run metadata and file hashes are preserved, and future reruns are not guaranteed to produce identical per-item scores.
- Post-attack F1 is computed as binary, with AI as the positive class and a threshold of 0.5. Members' historical F1 scores should not be assumed to share the exact same definition without verification.
- `data/E/run_metadata.json` records dependency versions, models, and interface file hashes from the original detection run, without containing API keys. The default verification dependencies below differ from model inference dependencies, and Jupyter is merely the tool used to open `.ipynb` files.

References: Project repository <https://github.com/ZiyingTeng/AI-Detector>; original run metadata and member export files (included in package); `scikit-learn` implementations of `roc_auc_score`, `f1_score`, and `roc_curve`. Explanations of AUROC in this document refer to the positive/negative binary classification setting.

In [11]:
print('Original detection environment record:')
show_table(pd.DataFrame([{'package': name, 'version': version} for name, version in metadata['versions'].items()]))
print('\nOriginal D run settings:', metadata['settings']['P7_D_MODEL'], metadata['settings']['P7_D_PROMPT'],
      'max_tokens=', metadata['D_request_policy']['max_tokens'])
print('Completed: input verification, 1,796 score verifications, metric recalculation, historical coverage check, demo pair verification, and export of four files.')


Original detection environment record:
     package version
       numpy   2.5.4
scikit-learn   1.9.1
     xgboost   3.2.0
       torch  2.15.0
transformers  5.22.1
      openai  3.49.0

Original D run settings: deepseek-flash few_shot max_tokens= 4096
Completed: input verification, 1,796 score verifications, metric recalculation, historical coverage check, demo pair verification, and export of four files.
